# Topic Modeling of Policy-Cited Literature

This notebook performs topic modeling on the cleaned publication
datasets produced by the data-preparation workflow.

The analysis includes text preprocessing, document-term matrix
construction, LDA model selection, final topic estimation, topic
interpretation, prevalence analysis, and temporal analysis.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
import subprocess
import tempfile

# Project directories
NOTEBOOK_DIR = Path.cwd()

if NOTEBOOK_DIR.name == "notebooks":
    PROJECT_DIR = NOTEBOOK_DIR.parent
else:
    PROJECT_DIR = NOTEBOOK_DIR

DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "output"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Project directory :", PROJECT_DIR)
print("Data directory    :", DATA_DIR)
print("Output directory  :", OUTPUT_DIR)

## 1. Load Cleaned Publication Data

The cleaned Overton and Scopus publication datasets produced by the
data-preparation notebook are loaded separately. The two sources are
retained as distinct datasets rather than merged.

In [2]:
OVERTON_FILE = (
    DATA_DIR / "overton_full_clean.xlsx"
)

SCOPUS_FILE = (
    DATA_DIR / "scopus_full_clean.xlsx"
)

overton = pd.read_excel(
    OVERTON_FILE
)

scopus = pd.read_excel(
    SCOPUS_FILE
)

print("Overton")
print("-------")
print(f"Documents : {len(overton):,}")
print(f"Columns   : {len(overton.columns):,}")

print("\nScopus")
print("------")
print(f"Documents : {len(scopus):,}")
print(f"Columns   : {len(scopus.columns):,}")

Overton
-------
Documents : 14,266
Columns   : 21

Scopus
------
Documents : 16,403
Columns   : 18


### 1.1 Define the Independent Topic-Modeling Corpora

The Overton and Scopus publication collections are analyzed as
independent corpora. Each corpus therefore receives its own text
preprocessing, document-term matrix, LDA model-selection procedure,
final topic model, and downstream topic analysis.

In [3]:
corpora = {
    "overton": overton.copy(),
    "scopus": scopus.copy(),
}

for corpus_name, corpus_df in corpora.items():

    print(f"{corpus_name.upper()}")
    print("-" * len(corpus_name))

    print(
        f"Documents          : "
        f"{len(corpus_df):,}"
    )

    print(
        f"Non-missing titles : "
        f"{corpus_df['Title'].notna().sum():,}"
    )

    print(
        f"Non-missing abstracts: "
        f"{corpus_df['Abstract'].notna().sum():,}"
    )

    print(
        f"Missing abstracts  : "
        f"{corpus_df['Abstract'].isna().sum():,}"
    )

    print()

OVERTON
-------
Documents          : 14,266
Non-missing titles : 14,266
Non-missing abstracts: 14,264
Missing abstracts  : 2

SCOPUS
------
Documents          : 16,403
Non-missing titles : 16,403
Non-missing abstracts: 16,403
Missing abstracts  : 0



## 2. Prepare Corpora for Topic Modeling

Topic modeling is performed independently for the Overton and Scopus
datasets. Publications without abstracts are excluded because the LDA
models are estimated from abstract text.

In [4]:
lda_corpora = {}

for corpus_name, corpus_df in corpora.items():

    lda_df = (
        corpus_df[
            corpus_df["Abstract"].notna()
        ]
        .copy()
        .reset_index(drop=True)
    )

    # Remove abstracts that are empty after whitespace stripping.
    lda_df["Abstract"] = (
        lda_df["Abstract"]
        .astype(str)
        .str.strip()
    )

    lda_df = (
        lda_df[
            lda_df["Abstract"] != ""
        ]
        .reset_index(drop=True)
    )

    lda_corpora[corpus_name] = lda_df

    print(corpus_name.upper())
    print("-" * len(corpus_name))
    print(
        f"Input publications : "
        f"{len(corpus_df):,}"
    )
    print(
        f"LDA documents      : "
        f"{len(lda_df):,}"
    )
    print(
        f"Excluded           : "
        f"{len(corpus_df) - len(lda_df):,}"
    )
    print()
    

OVERTON
-------
Input publications : 14,266
LDA documents      : 14,264
Excluded           : 2

SCOPUS
------
Input publications : 16,403
LDA documents      : 16,403
Excluded           : 0



## 3. R Text-Processing Backend

The abstract corpora are preprocessed using R `tm` and `SnowballC`
through `Rscript`. This preserves the text-processing methodology used
for the LDA analysis while allowing the complete workflow to be
controlled from Python.

In [5]:
from pathlib import Path
import subprocess

RSCRIPT = Path(
    r"C:\Program Files\R\R-4.5.2\bin\Rscript.exe"
)

if not RSCRIPT.exists():
    raise FileNotFoundError(
        f"Rscript not found: {RSCRIPT}"
    )

# Check required R packages.
r_package_check = subprocess.run(
    [
        str(RSCRIPT),
        "-e",
        (
            'pkgs <- c("tm", "SnowballC", "slam", "topicmodels"); '
            'ok <- sapply(pkgs, requireNamespace, quietly=TRUE); '
            'cat(paste(pkgs, ok, sep="="), sep="\\n")'
        ),
    ],
    capture_output=True,
    text=True,
    check=True,
)

print("Rscript:")
print(RSCRIPT)

print("\nRequired R packages:")
print(r_package_check.stdout)

Rscript:
C:\Program Files\R\R-4.5.2\bin\Rscript.exe

Required R packages:
tm=TRUE
SnowballC=TRUE
slam=TRUE
topicmodels=TRUE



### 3.1 Text-Preprocessing Configuration

The Overton and Scopus corpora are processed using an identical text
preprocessing configuration to support direct comparison between the
two independently estimated topic models.

Standard English stopwords are supplemented with terms appearing
explicitly in the literature-search query because these terms define
the corpus but provide limited information for distinguishing latent
topics.

In [6]:
# Search-query terms removed from both corpora.
# change this based on your topic

QUERY_STOPWORDS = [
    # Search-query terms
    "power",
    "flow",
    "machine",
    "learning",
    "optimization",
    "optimisation",

    # Publisher/copyright boilerplate
    "©",
]

# Initial DTM configuration.
MIN_TERM_LENGTH = 3
MIN_DOC_FREQ = 3

print("Query-specific stopwords:")
for word in QUERY_STOPWORDS:
    print(f"  - {word}")

print("\nDTM configuration")
print("-----------------")
print("Minimum term length     :", MIN_TERM_LENGTH)
print("Minimum document freq.  :", MIN_DOC_FREQ)

Query-specific stopwords:
  - power
  - flow
  - machine
  - learning
  - optimization
  - optimisation
  - ©

DTM configuration
-----------------
Minimum term length     : 3
Minimum document freq.  : 3


### 3.2 Preprocess Abstracts with R `tm`

The same R `tm` and `SnowballC` preprocessing pipeline is applied
independently to the Overton and Scopus abstracts. Processing includes
lowercasing, punctuation and number removal, whitespace normalization,
English and query-specific stopword removal, and English Snowball
stemming.

The resulting corpora are used to inspect vocabulary characteristics
before the final document-term matrices are constructed.

In [7]:
R_PREPROCESS_DIR = OUTPUT_DIR / "r_preprocessing"

R_PREPROCESS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

R_PREPROCESS_SCRIPT = (
    R_PREPROCESS_DIR / "preprocess_corpus.R"
)

r_preprocess_code = r'''
args <- commandArgs(trailingOnly = TRUE)

input_file  <- args[1]
output_file <- args[2]

suppressPackageStartupMessages({
    library(tm)
    library(SnowballC)
})

# ------------------------------------------------------------
# Load abstracts
# ------------------------------------------------------------

data <- read.csv(
    input_file,
    stringsAsFactors = FALSE,
    check.names = FALSE,
    fileEncoding = "UTF-8"
)

abstracts <- data$Abstract

# ------------------------------------------------------------
# Shared stopwords
# ------------------------------------------------------------

custom_stops <- c(
    "power",
    "flow",
    "machine",
    "learning",
    "optimization",
    "optimisation",
    "©"
)

all_stops <- unique(
    c(
        tm::stopwords("english"),
        custom_stops
    )
)

# ------------------------------------------------------------
# tm preprocessing
# ------------------------------------------------------------

corpus <- VCorpus(
    VectorSource(abstracts)
)

corpus <- tm_map(
    corpus,
    content_transformer(tolower)
)

corpus <- tm_map(
    corpus,
    removePunctuation
)

corpus <- tm_map(
    corpus,
    removeNumbers
)

corpus <- tm_map(
    corpus,
    stripWhitespace
)

corpus <- tm_map(
    corpus,
    removeWords,
    all_stops
)

# Remove copyright symbol explicitly.
corpus <- tm_map(
    corpus,
    content_transformer(
        function(x) gsub(
            "©",
            " ",
            x,
            fixed = TRUE
        )
    )
)

corpus <- tm_map(
    corpus,
    stripWhitespace
)

corpus <- tm_map(
    corpus,
    stemDocument,
    language = "english"
)

corpus <- tm_map(
    corpus,
    stripWhitespace
)

processed <- vapply(
    corpus,
    as.character,
    character(1)
)

result <- data.frame(
    document_id = seq_along(processed),
    processed_text = processed,
    stringsAsFactors = FALSE
)

write.csv(
    result,
    output_file,
    row.names = FALSE,
    fileEncoding = "UTF-8"
)
'''

R_PREPROCESS_SCRIPT.write_text(
    r_preprocess_code,
    encoding="utf-8",
)

print(
    "Created R preprocessing script:",
    R_PREPROCESS_SCRIPT
)

Created R preprocessing script: m:\projects_latex\review_paper_2\output\r_preprocessing\preprocess_corpus.R


In [ ]:
processed_corpora = {}

for corpus_name, corpus_df in lda_corpora.items():

    input_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_abstracts.csv"
    )

    output_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_processed.csv"
    )

    # ---------------------------------------------------------
    # Use cached processed corpus if it already exists
    # ---------------------------------------------------------

    if output_file.exists():

        print(
            f"Loading cached {corpus_name.upper()} "
            f"processed corpus ..."
        )

    else:

        print(
            f"Processing {corpus_name.upper()} with R ..."
        )

        # Export abstracts only when preprocessing is required.
        corpus_df[
            ["Abstract"]
        ].to_csv(
            input_file,
            index=False,
            encoding="utf-8",
        )

        run = subprocess.run(
            [
                str(RSCRIPT),
                str(R_PREPROCESS_SCRIPT),
                str(input_file),
                str(output_file),
            ],
            capture_output=True,
            text=True,
            check=True,
        )

    # ---------------------------------------------------------
    # Load processed corpus
    # ---------------------------------------------------------

    processed_df = pd.read_csv(
        output_file,
        keep_default_na=False,
    )

    processed_corpora[
        corpus_name
    ] = processed_df

    print(
        f"Documents processed : "
        f"{len(processed_df):,}"
    )

    print(
        f"Empty documents     : "
        f"{(processed_df['processed_text'].str.strip() == '').sum():,}"
    )

    print()

Processing OVERTON ...
Documents processed : 14,264
Empty documents     : 1

Processing SCOPUS ...
Documents processed : 16,403
Empty documents     : 0



In [9]:
for corpus_name, processed_df in processed_corpora.items():

    token_counts = (
        processed_df[
            "processed_text"
        ]
        .str.split()
        .str.len()
    )

    print(corpus_name.upper())
    print("-" * len(corpus_name))

    print(
        f"Documents       : "
        f"{len(processed_df):,}"
    )

    print(
        f"Total tokens    : "
        f"{int(token_counts.sum()):,}"
    )

    print(
        f"Minimum tokens  : "
        f"{int(token_counts.min()):,}"
    )

    print(
        f"Median tokens   : "
        f"{token_counts.median():.0f}"
    )

    print(
        f"Mean tokens     : "
        f"{token_counts.mean():.1f}"
    )

    print(
        f"Maximum tokens  : "
        f"{int(token_counts.max()):,}"
    )

    print()

OVERTON
-------
Documents       : 14,264
Total tokens    : 1,572,042
Minimum tokens  : 0
Median tokens   : 107
Mean tokens     : 110.2
Maximum tokens  : 565

SCOPUS
------
Documents       : 16,403
Total tokens    : 2,058,934
Minimum tokens  : 2
Median tokens   : 123
Mean tokens     : 125.5
Maximum tokens  : 452



### 3.3 Inspect Frequent Terms

The most frequent terms remaining after preprocessing are inspected
before constructing the final document-term matrices. This diagnostic
is used to identify high-frequency generic terms that provide little
thematic discrimination and may therefore warrant inclusion in the
shared custom stopword list.

In [10]:
from collections import Counter

term_frequency_tables = {}

for corpus_name, processed_df in processed_corpora.items():

    term_frequency = Counter(
        token
        for text in processed_df["processed_text"]
        for token in text.split()
    )

    top_terms = pd.DataFrame(
        term_frequency.most_common(50),
        columns=[
            "term",
            "frequency",
        ],
    )

    term_frequency_tables[
        corpus_name
    ] = top_terms

    print(f"\n{corpus_name.upper()}")
    print("-" * len(corpus_name))

    display(
        top_terms.head(30)
    )


OVERTON
-------


,term,frequency
0,use,15161
1,system,14469
2,model,13987
3,energi,9977
4,can,7645
5,data,7553
6,result,7509
7,studi,7246
8,paper,6678
9,develop,6644



SCOPUS
------


,term,frequency
0,system,32854
1,energi,22024
2,propos,21979
3,algorithm,21225
4,model,21219
5,method,18906
6,use,17585
7,network,14453
8,distribut,13961
9,control,13821


## 4. Document-Term Matrix Construction

Separate document-term matrices are constructed for the Overton and
Scopus corpora using R `tm`. The same preprocessing and vocabulary
filtering rules are applied to both corpora.

Terms shorter than three characters and terms occurring in fewer than
three documents are excluded. The resulting matrix dimensions and
sparsity are inspected before LDA model selection.

In [11]:
R_DTM_SCRIPT = (
    R_PREPROCESS_DIR / "construct_dtm.R"
)

r_dtm_code = r'''
args <- commandArgs(trailingOnly = TRUE)

input_file   <- args[1]
summary_file <- args[2]
terms_file   <- args[3]

suppressPackageStartupMessages({
    library(tm)
    library(slam)
})

MIN_TERM_LENGTH <- 3
MIN_DOC_FREQ <- 25

# ------------------------------------------------------------
# Load processed documents
# ------------------------------------------------------------

data <- read.csv(
    input_file,
    stringsAsFactors = FALSE,
    check.names = FALSE,
    fileEncoding = "UTF-8"
)

processed <- data$processed_text

# Remove empty documents before DTM construction.
valid <- !is.na(processed) & nzchar(trimws(processed))

processed <- processed[valid]

# ------------------------------------------------------------
# Construct tm corpus
# ------------------------------------------------------------

corpus <- VCorpus(
    VectorSource(processed)
)

# ------------------------------------------------------------
# Initial DTM
# ------------------------------------------------------------

dtm <- DocumentTermMatrix(
    corpus,
    control = list(
        wordLengths = c(
            MIN_TERM_LENGTH,
            Inf
        )
    )
)

# ------------------------------------------------------------
# Minimum document frequency
# ------------------------------------------------------------

doc_freq <- slam::col_sums(
    dtm > 0
)

keep <- doc_freq >= MIN_DOC_FREQ

dtm <- dtm[
    ,
    keep
]

doc_freq <- doc_freq[
    keep
]

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

n_docs <- dtm$nrow
n_terms <- dtm$ncol
n_nonzero <- length(dtm$v)
n_tokens <- sum(dtm$v)

density <- (
    n_nonzero
    / (n_docs * n_terms)
    * 100
)

summary_result <- data.frame(
    Documents = n_docs,
    Vocabulary = n_terms,
    Nonzero_entries = n_nonzero,
    Total_tokens = n_tokens,
    Density_percent = density
)

write.csv(
    summary_result,
    summary_file,
    row.names = FALSE
)

term_result <- data.frame(
    Term = Terms(dtm),
    Document_frequency = as.numeric(doc_freq),
    stringsAsFactors = FALSE
)

write.csv(
    term_result,
    terms_file,
    row.names = FALSE
)
'''

R_DTM_SCRIPT.write_text(
    r_dtm_code,
    encoding="utf-8",
)

print(
    "Created R DTM script:",
    R_DTM_SCRIPT
)

Created R DTM script: m:\projects_latex\review_paper_2\output\r_preprocessing\construct_dtm.R


In [ ]:
dtm_summaries = {}
dtm_term_tables = {}

for corpus_name in [
    "overton",
    "scopus",
]:

    processed_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_processed.csv"
    )

    summary_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_dtm_summary.csv"
    )

    terms_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_dtm_terms.csv"
    )

    subprocess.run(
        [
            str(RSCRIPT),
            str(R_DTM_SCRIPT),
            str(processed_file),
            str(summary_file),
            str(terms_file),
        ],
        capture_output=True,
        text=True,
        check=True,
    )

    summary_df = pd.read_csv(
        summary_file
    )

    terms_df = pd.read_csv(
        terms_file
    )

    dtm_summaries[
        corpus_name
    ] = summary_df

    dtm_term_tables[
        corpus_name
    ] = terms_df

    row = summary_df.iloc[0]

    print(corpus_name.upper())
    print("-" * len(corpus_name))

    print(
        f"Documents       : "
        f"{int(row['Documents']):,}"
    )

    print(
        f"Vocabulary      : "
        f"{int(row['Vocabulary']):,}"
    )

    print(
        f"Nonzero entries : "
        f"{int(row['Nonzero_entries']):,}"
    )

    print(
        f"Total tokens    : "
        f"{int(row['Total_tokens']):,}"
    )

    print(
        f"Matrix density  : "
        f"{row['Density_percent']:.4f}%"
    )

    print()

### 4.1 Vocabulary Filtering

Given the large corpus sizes, a minimum document-frequency threshold
of 25 documents is applied identically to both datasets. This removes
very rare terms while retaining vocabulary occurring in approximately
0.18% or more of Overton documents and 0.15% or more of Scopus
documents.

The resulting vocabularies contain 3,589 terms for Overton and 3,288
terms for Scopus.

In [ ]:
# Compare vocabulary sizes under alternative document-frequency
# thresholds using the document frequencies already calculated in R.

DF_THRESHOLDS = [
    3,
    5,
    10,
    20,
    25,
    50,
    100,
]

df_threshold_results = []

for corpus_name, terms_df in dtm_term_tables.items():

    for threshold in DF_THRESHOLDS:

        retained = (
            terms_df["Document_frequency"]
            >= threshold
        ).sum()

        df_threshold_results.append({
            "Corpus": corpus_name.capitalize(),
            "Min_Document_Frequency": threshold,
            "Retained_Terms": int(retained),
        })

df_threshold_results = pd.DataFrame(
    df_threshold_results
)

df_threshold_pivot = (
    df_threshold_results
    .pivot(
        index="Min_Document_Frequency",
        columns="Corpus",
        values="Retained_Terms",
    )
)

df_threshold_pivot

In [ ]:
for corpus_name, terms_df in dtm_term_tables.items():

    print(corpus_name.upper())
    print("-" * len(corpus_name))

    n_docs = int(
        dtm_summaries[
            corpus_name
        ].iloc[0]["Documents"]
    )

    for threshold in DF_THRESHOLDS:

        retained = (
            terms_df["Document_frequency"]
            >= threshold
        ).sum()

        print(
            f"DF >= {threshold:3d} "
            f"({threshold / n_docs * 100:5.3f}% docs)"
            f" : {retained:6,d} terms"
        )

    print()

## 5. LDA Topic Modeling and Model Selection

LDA models are estimated independently for the Overton and Scopus
corpora using Gibbs sampling with R `topicmodels`.

For model selection, each corpus is divided reproducibly into 80%
training and 20% held-out documents using a fixed random seed. The
held-out data are subsequently used to evaluate candidate topic
numbers using perplexity.

### 5.1 Training and Held-Out Splits

An independent 80/20 split is generated for each corpus using R's
random-number generator with seed 123. The same sampling procedure is
therefore applied to both datasets.

In [ ]:
RANDOM_SEED = 123

lda_splits = {}

for corpus_name, summary_df in dtm_summaries.items():

    n_documents = int(
        summary_df.iloc[0]["Documents"]
    )

    split_result = subprocess.run(
        [
            str(RSCRIPT),
            "-e",
            (
                f"set.seed({RANDOM_SEED}); "
                f"n <- {n_documents}; "
                "train_id <- sample("
                "seq_len(n), "
                "size=floor(0.8*n), "
                "replace=FALSE"
                "); "
                'cat(train_id, sep=",")'
            ),
        ],
        capture_output=True,
        text=True,
        check=True,
    )

    # Keep R's 1-based indices because these will
    # subsequently be passed back to R topicmodels.
    train_id = np.fromstring(
        split_result.stdout.strip(),
        sep=",",
        dtype=int,
    )

    all_id = np.arange(
        1,
        n_documents + 1
    )

    test_id = np.setdiff1d(
        all_id,
        train_id
    )

    lda_splits[corpus_name] = {
        "train_id": train_id,
        "test_id": test_id,
    }

    print(corpus_name.upper())
    print("-" * len(corpus_name))

    print(
        f"Total documents    : "
        f"{n_documents:,}"
    )

    print(
        f"Training documents : "
        f"{len(train_id):,}"
    )

    print(
        f"Held-out documents : "
        f"{len(test_id):,}"
    )

    print(
        f"Overlap             : "
        f"{len(np.intersect1d(train_id, test_id))}"
    )

    print()

### 5.2 Gibbs-LDA Runtime Benchmark

Before evaluating a range of candidate topic numbers, a single
\(K=15\) model is fitted to each training corpus to measure the
computational cost of Gibbs sampling on the two large document-term
matrices.

This benchmark is used only to estimate runtime and does not determine
the final number of topics.

In [ ]:
LDA_DIR = OUTPUT_DIR / "lda"

LDA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

for corpus_name, split in lda_splits.items():

    split_file = (
        LDA_DIR
        / f"{corpus_name}_train_indices.csv"
    )

    pd.DataFrame({
        "train_id": split["train_id"]
    }).to_csv(
        split_file,
        index=False,
    )

print("Training-index files saved.")

In [ ]:
R_LDA_BENCHMARK_SCRIPT = (
    LDA_DIR / "benchmark_lda.R"
)

r_lda_benchmark_code = r'''
args <- commandArgs(trailingOnly = TRUE)

processed_file <- args[1]
train_file     <- args[2]
output_file    <- args[3]

suppressPackageStartupMessages({
    library(tm)
    library(slam)
    library(topicmodels)
})

MIN_TERM_LENGTH <- 3
MIN_DOC_FREQ <- 25

# ------------------------------------------------------------
# Load processed documents
# ------------------------------------------------------------

data <- read.csv(
    processed_file,
    stringsAsFactors = FALSE,
    check.names = FALSE,
    fileEncoding = "UTF-8"
)

processed <- data$processed_text

valid <- (
    !is.na(processed)
    & nzchar(trimws(processed))
)

processed <- processed[valid]

corpus <- VCorpus(
    VectorSource(processed)
)

# ------------------------------------------------------------
# Reconstruct final corpus DTM
# ------------------------------------------------------------

dtm <- DocumentTermMatrix(
    corpus,
    control = list(
        wordLengths = c(
            MIN_TERM_LENGTH,
            Inf
        )
    )
)

doc_freq <- slam::col_sums(
    dtm > 0
)

dtm <- dtm[
    ,
    doc_freq >= MIN_DOC_FREQ
]

# ------------------------------------------------------------
# Train / held-out split
# ------------------------------------------------------------

train_id <- read.csv(
    train_file
)$train_id

test_id <- setdiff(
    seq_len(dtm$nrow),
    train_id
)

dtm_train <- dtm[
    train_id,
]

dtm_test <- dtm[
    test_id,
]

# Remove terms absent from the training corpus.
train_terms <- (
    slam::col_sums(
        dtm_train > 0
    ) > 0
)

dtm_train <- dtm_train[
    ,
    train_terms
]

dtm_test <- dtm_test[
    ,
    train_terms
]

# ------------------------------------------------------------
# Remove documents empty after vocabulary filtering
# ------------------------------------------------------------

train_nonempty <- (
    slam::row_sums(dtm_train) > 0
)

test_nonempty <- (
    slam::row_sums(dtm_test) > 0
)

n_empty_train <- sum(!train_nonempty)
n_empty_test <- sum(!test_nonempty)

dtm_train <- dtm_train[
    train_nonempty,
]

dtm_test <- dtm_test[
    test_nonempty,
]

cat(
    "Empty training documents removed:",
    n_empty_train,
    "\n"
)

cat(
    "Empty held-out documents removed:",
    n_empty_test,
    "\n"
)

cat(
    "Final training documents:",
    dtm_train$nrow,
    "\n"
)

cat(
    "Final held-out documents:",
    dtm_test$nrow,
    "\n"
)

# ------------------------------------------------------------
# Benchmark K = 15
# ------------------------------------------------------------

k <- 15

start_time <- Sys.time()

lda_model <- topicmodels::LDA(
    dtm_train,
    k = k,
    method = "Gibbs",
    control = list(
    seed = 123,
    burnin = 50,
    iter = 100,
    thin = 10
)
)

heldout_perplexity <- topicmodels::perplexity(
    lda_model,
    newdata = dtm_test
)

elapsed <- as.numeric(
    difftime(
        Sys.time(),
        start_time,
        units = "secs"
    )
)

result <- data.frame(
    K = k,
    Training_documents = dtm_train$nrow,
    Heldout_documents = dtm_test$nrow,
    Training_vocabulary = dtm_train$ncol,
    Training_tokens = sum(dtm_train$v),
    Heldout_tokens = sum(dtm_test$v),
    Heldout_perplexity = heldout_perplexity,
    Elapsed_seconds = elapsed
)

write.csv(
    result,
    output_file,
    row.names = FALSE
)
'''

R_LDA_BENCHMARK_SCRIPT.write_text(
    r_lda_benchmark_code,
    encoding="utf-8",
)

print(
    "Created:",
    R_LDA_BENCHMARK_SCRIPT
)

In [ ]:
corpus_name = "overton"

processed_file = (
    R_PREPROCESS_DIR
    / f"{corpus_name}_processed.csv"
)

train_file = (
    LDA_DIR
    / f"{corpus_name}_train_indices.csv"
)

benchmark_output = (
    LDA_DIR
    / f"{corpus_name}_k15_benchmark.csv"
)

print("Running Overton K=15 benchmark...")

benchmark_run = subprocess.run(
    [
        str(RSCRIPT),
        str(R_LDA_BENCHMARK_SCRIPT),
        str(processed_file),
        str(train_file),
        str(benchmark_output),
    ],
    capture_output=True,
    text=True,
    check=False,
)

print("Return code:", benchmark_run.returncode)

print("\nR stdout:")
print(
    benchmark_run.stdout
    if benchmark_run.stdout.strip()
    else "(empty)"
)

print("\nR stderr:")
print(
    benchmark_run.stderr
    if benchmark_run.stderr.strip()
    else "(empty)"
)